# Step 5: Python Exploratory Analysis

This notebook analyzes the processed funnel datasets for:
- funnel progression and conversion
- payment and ticketing failures
- refund behavior
- revenue at risk

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import duckdb
from pathlib import Path

plt.style.use('ggplot')
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 140)

In [5]:
# Resolve project root robustly, even when notebook CWD is notebooks/.
cwd = Path.cwd().resolve()

candidate_roots = [cwd] + list(cwd.parents)
project_root = None
for candidate in candidate_roots:
    if (candidate / 'data' / 'processed' / 'search_booking_events.csv').exists():
        project_root = candidate
        break

if project_root is None:
    raise FileNotFoundError(
        'Could not locate project root containing data/processed/search_booking_events.csv'
    )

data_dir = project_root / 'data' / 'processed'

search_booking_path = data_dir / 'search_booking_events.csv'
payment_path = data_dir / 'payment_events.csv'
ticket_path = data_dir / 'ticket_events.csv'
refund_path = data_dir / 'refund_events.csv'

search_booking = pd.read_csv(search_booking_path, parse_dates=['date_time', 'srch_ci', 'srch_co'])
payment = pd.read_csv(payment_path, parse_dates=['payment_attempt_time'])
ticket = pd.read_csv(ticket_path, parse_dates=['ticket_issued_time'])
refund = pd.read_csv(refund_path, parse_dates=['refund_request_time'])

print(f'Project root: {project_root}')
print('Loaded datasets:')
print(f'- search_booking: {search_booking.shape}')
print(f'- payment:       {payment.shape}')
print(f'- ticket:        {ticket.shape}')
print(f'- refund:        {refund.shape}')

Project root: C:\Users\eniko\Documents\coding_projects\flights-funnel-payments-dashboard
Loaded datasets:
- search_booking: (100000, 26)
- payment:       (8122, 12)
- ticket:        (8122, 7)
- refund:        (7402, 8)


In [6]:
# Validate row counts and key relationships across tables.
searches = len(search_booking)
bookings = int(search_booking.loc[search_booking['is_booking'] == 1, 'cnt'].sum())
payment_attempts = len(payment)
successful_payments = int((payment['payment_status'] == 'success').sum())
issued_tickets = int((ticket['ticket_status'] == 'issued').sum())
refund_requests = int(refund['refund_requested'].fillna(False).sum())

print('Row-count checks:')
print(f'- Searches:           {searches:,}')
print(f'- Bookings (weighted):{bookings:,}')
print(f'- Payment attempts:   {payment_attempts:,}')
print(f'- Successful payments:{successful_payments:,}')
print(f'- Issued tickets:     {issued_tickets:,}')
print(f'- Refund requests:    {refund_requests:,}')

print('\nRelationship checks:')
print(f"- payment.booking_id unique: {payment['booking_id'].nunique():,}")
print(f"- ticket.payment_id unmatched in payment: {ticket.loc[~ticket['payment_id'].isin(payment['payment_id'])].shape[0]}")
print(f"- refund.payment_id unmatched in payment: {refund.loc[~refund['payment_id'].isin(payment['payment_id'])].shape[0]}")
print(f"- failed payments with issued tickets: {ticket.merge(payment[['payment_id', 'payment_status']], on='payment_id', how='left').query('payment_status == \"failed\" and ticket_status == \"issued\"').shape[0]}")

Row-count checks:
- Searches:           100,000
- Bookings (weighted):8,122
- Payment attempts:   8,122
- Successful payments:7,402
- Issued tickets:     7,147
- Refund requests:    380

Relationship checks:
- payment.booking_id unique: 8,122
- ticket.payment_id unmatched in payment: 0
- refund.payment_id unmatched in payment: 0
- failed payments with issued tickets: 0


In [7]:
# Calculate funnel metrics, conversion rates, and drop-off rates.
funnel_df = pd.DataFrame({
    'stage': [
        'searches',
        'bookings',
        'payment_attempts',
        'successful_payments',
        'issued_tickets',
        'refunds'
    ],
    'count': [
        searches,
        bookings,
        payment_attempts,
        successful_payments,
        issued_tickets,
        refund_requests
    ]
})

funnel_df['conversion_from_search_pct'] = (funnel_df['count'] / searches * 100).round(2)
funnel_df['prev_count'] = funnel_df['count'].shift(1)
funnel_df['step_conversion_pct'] = (funnel_df['count'] / funnel_df['prev_count'] * 100).round(2)
funnel_df['step_dropoff_pct'] = (100 - funnel_df['step_conversion_pct']).round(2)

funnel_df.loc[0, ['step_conversion_pct', 'step_dropoff_pct']] = np.nan
funnel_df

,stage,count,conversion_from_search_pct,prev_count,step_conversion_pct,step_dropoff_pct
0,searches,100000,100.00,NaN,NaN,NaN
1,bookings,8122,8.12,100000.0,8.12,91.88
2,payment_attempts,8122,8.12,8122.0,100.00,0.00
3,successful_payments,7402,7.40,8122.0,91.14,8.86
4,issued_tickets,7147,7.15,7402.0,96.55,3.45
5,refunds,380,0.38,7147.0,5.32,94.68


In [8]:
# Analyze payment failures by method, device, country, and error code.
payment_failure_by_method_device = (
    payment.groupby(['device', 'payment_method'], dropna=False)
    .agg(
        payment_attempts=('payment_id', 'count'),
        payment_failures=('payment_status', lambda s: (s == 'failed').sum()),
        payment_successes=('payment_status', lambda s: (s == 'success').sum())
    )
    .reset_index()
)
payment_failure_by_method_device['failure_rate_pct'] = (
    payment_failure_by_method_device['payment_failures']
    / payment_failure_by_method_device['payment_attempts'] * 100
).round(2)

payment_failure_by_country = (
    payment.groupby('country', dropna=False)
    .agg(
        payment_attempts=('payment_id', 'count'),
        payment_failures=('payment_status', lambda s: (s == 'failed').sum())
    )
    .reset_index()
)
payment_failure_by_country['failure_rate_pct'] = (
    payment_failure_by_country['payment_failures']
    / payment_failure_by_country['payment_attempts'] * 100
).round(2)
payment_failure_by_country = payment_failure_by_country.sort_values('payment_failures', ascending=False)

payment_failure_by_error_code = (
    payment.loc[payment['payment_status'] == 'failed']
    .groupby('payment_error_code', dropna=False)
    .size()
    .reset_index(name='failure_count')
    .sort_values('failure_count', ascending=False)
)

print('Payment failure by method/device:')
display(payment_failure_by_method_device.sort_values('failure_rate_pct', ascending=False).head(15))

print('Payment failure by country (top by failure count):')
display(payment_failure_by_country.head(15))

print('Payment failure by error code:')
display(payment_failure_by_error_code)

Payment failure by method/device:


,device,payment_method,payment_attempts,payment_failures,payment_successes,failure_rate_pct
8,mobile,credit_card,325,42,283,12.92
6,mobile,apple_pay,70,9,61,12.86
7,mobile,bank_transfer,70,8,62,11.43
1,desktop,bank_transfer,632,61,571,9.65
3,desktop,debit_card,1557,140,1417,8.99
2,desktop,credit_card,2843,251,2592,8.83
4,desktop,google_pay,625,55,570,8.80
9,mobile,debit_card,174,15,159,8.62
0,desktop,apple_pay,687,58,629,8.44
10,mobile,google_pay,64,5,59,7.81


Payment failure by country (top by failure count):


,country,payment_attempts,payment_failures,failure_rate_pct
42,66,4594,405,8.82
108,205,883,70,7.93
2,3,428,43,10.05
44,69,407,36,8.85
27,46,169,21,12.43
1,1,145,16,11.03
48,77,156,14,8.97
114,215,119,10,8.40
74,133,69,9,13.04
0,0,37,8,21.62


Payment failure by error code:


,payment_error_code,failure_count
2,CARD_DECLINED,242
3,INSUFFICIENT_FUNDS,171
1,AUTH_TIMEOUT,104
4,PAYMENT_PROVIDER_ERROR,103
0,3DS_FAILED,100


In [9]:
# Revenue at risk from failed payments.
failed_payment_revenue = payment.loc[payment['payment_status'] == 'failed', 'amount'].sum()
attempted_payment_revenue = payment['amount'].sum()
successful_payment_revenue = payment.loc[payment['payment_status'] == 'success', 'amount'].sum()

revenue_risk_df = pd.DataFrame({
    'metric': ['attempted_payment_revenue', 'failed_payment_revenue', 'successful_payment_revenue'],
    'amount_usd': [attempted_payment_revenue, failed_payment_revenue, successful_payment_revenue]
})
revenue_risk_df['amount_usd'] = revenue_risk_df['amount_usd'].round(2)

failed_revenue_share_pct = round((failed_payment_revenue / attempted_payment_revenue) * 100, 2)
print(f'Failed payment revenue at risk: ${failed_payment_revenue:,.2f} ({failed_revenue_share_pct}% of attempted payment revenue)')
revenue_risk_df

Failed payment revenue at risk: $322,331.50 (8.89% of attempted payment revenue)


,metric,amount_usd
0,attempted_payment_revenue,3627247.19
1,failed_payment_revenue,322331.50
2,successful_payment_revenue,3304915.69


In [10]:
# Analyze ticketing failures after successful payment.
ticket_with_payment = ticket.merge(
    payment[['payment_id', 'payment_status', 'payment_method', 'device', 'country']],
    on='payment_id',
    how='left'
)

ticket_after_success = ticket_with_payment.loc[ticket_with_payment['payment_status'] == 'success'].copy()
ticket_outcome_summary = (
    ticket_after_success.groupby(['ticket_status', 'ticketing_error_code'], dropna=False)
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
)

ticket_failure_rate_pct = round((ticket_after_success['ticket_status'] != 'issued').mean() * 100, 2)
print(f'Ticket non-issuance rate after successful payment: {ticket_failure_rate_pct}%')
display(ticket_outcome_summary)

Ticket non-issuance rate after successful payment: 3.45%


,ticket_status,ticketing_error_code,count
4,issued,NONE,7147
0,failed,AIRLINE_CONFIRMATION_FAILED,47
5,pending,AIRLINE_CONFIRMATION_FAILED,44
8,pending,TICKETING_TIMEOUT,40
7,pending,SUPPLIER_ERROR,36
3,failed,TICKETING_TIMEOUT,26
6,pending,INVENTORY_MISMATCH,24
2,failed,SUPPLIER_ERROR,23
1,failed,INVENTORY_MISMATCH,15


In [11]:
# Analyze refund patterns.
refund_summary = (
    refund.groupby(['refund_requested', 'refund_status', 'refund_reason'], dropna=False)
    .agg(
        records=('refund_id', 'count'),
        total_refund_amount=('refund_amount', 'sum')
    )
    .reset_index()
    .sort_values(['refund_requested', 'records'], ascending=[False, False])
)

approved_refunds = refund.loc[refund['refund_status'] == 'approved', 'refund_amount'].sum()
pending_refunds = refund.loc[refund['refund_status'] == 'pending', 'refund_amount'].sum()
requested_refund_count = int(refund['refund_requested'].sum())

print(f'Refund requests: {requested_refund_count:,}')
print(f'Approved refund amount: ${approved_refunds:,.2f}')
print(f'Pending refund amount: ${pending_refunds:,.2f}')
display(refund_summary.head(20))

Refund requests: 380
Approved refund amount: $100,270.94
Pending refund amount: $16,122.20


,refund_requested,refund_status,refund_reason,records,total_refund_amount
2,True,approved,customer_cancelled,100,35250.60
1,True,approved,airline_schedule_change,63,26555.88
5,True,approved,ticketing_failure,46,21172.84
3,True,approved,duplicate_booking,29,9353.30
12,True,rejected,customer_cancelled,27,0.00
4,True,approved,payment_dispute,24,7938.32
7,True,pending,customer_cancelled,19,7412.16
10,True,pending,ticketing_failure,12,2858.06
13,True,rejected,duplicate_booking,12,0.00
14,True,rejected,payment_dispute,12,0.00


In [13]:
# Optional: use DuckDB directly in-notebook for a quick validation query.
con = duckdb.connect(database=':memory:')

search_file = (project_root / 'data' / 'processed' / 'search_booking_events.csv').as_posix()
payment_file = (project_root / 'data' / 'processed' / 'payment_events.csv').as_posix()
ticket_file = (project_root / 'data' / 'processed' / 'ticket_events.csv').as_posix()
refund_file = (project_root / 'data' / 'processed' / 'refund_events.csv').as_posix()

duckdb_query = f"""
WITH search_base AS (
    SELECT * FROM read_csv_auto('{search_file}', HEADER=TRUE)
),
payment_base AS (
    SELECT * FROM read_csv_auto('{payment_file}', HEADER=TRUE)
),
ticket_base AS (
    SELECT * FROM read_csv_auto('{ticket_file}', HEADER=TRUE)
),
refund_base AS (
    SELECT * FROM read_csv_auto('{refund_file}', HEADER=TRUE)
)
SELECT
    (SELECT COUNT(*) FROM search_base) AS searches,
    (SELECT SUM(CASE WHEN is_booking = 1 THEN cnt ELSE 0 END) FROM search_base) AS bookings,
    (SELECT COUNT(*) FROM payment_base) AS payment_attempts,
    (SELECT SUM(CASE WHEN payment_status = 'success' THEN 1 ELSE 0 END) FROM payment_base) AS successful_payments,
    (SELECT SUM(CASE WHEN ticket_status = 'issued' THEN 1 ELSE 0 END) FROM ticket_base) AS issued_tickets,
    (SELECT SUM(CASE WHEN refund_requested THEN 1 ELSE 0 END) FROM refund_base) AS refunds
"""

duckdb_funnel = con.execute(duckdb_query).fetchdf()
con.close()
duckdb_funnel

,searches,bookings,payment_attempts,successful_payments,issued_tickets,refunds
0,100000,8122.0,8122,7402.0,7147.0,380.0


In [ ]:
# Chart 1: funnel counts.
plt.figure(figsize=(10, 5))
plt.bar(funnel_df['stage'], funnel_df['count'], color=['#4C78A8', '#59A14F', '#F28E2B', '#E15759', '#76B7B2', '#EDC948'])
plt.title('Funnel Counts')
plt.ylabel('Count')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
# Chart 2: payment failure rates by method and device (top 10 combinations by attempts).
top_failure_chart = payment_failure_by_method_device.sort_values('payment_attempts', ascending=False).head(10).copy()
top_failure_chart['method_device'] = top_failure_chart['device'] + ' | ' + top_failure_chart['payment_method']

plt.figure(figsize=(11, 5))
plt.bar(top_failure_chart['method_device'], top_failure_chart['failure_rate_pct'], color='#E15759')
plt.title('Payment Failure Rate by Method and Device (Top 10 by Attempts)')
plt.ylabel('Failure Rate (%)')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Chart 3: top payment error codes.
top_errors = payment_failure_by_error_code.head(10)

plt.figure(figsize=(9, 5))
plt.bar(top_errors['payment_error_code'].astype(str), top_errors['failure_count'], color='#F28E2B')
plt.title('Top Payment Failure Error Codes')
plt.ylabel('Failure Count')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
# Chart 4: revenue at risk from failed payments vs successful payment revenue.
risk_plot_df = revenue_risk_df.copy()
risk_plot_df = risk_plot_df[risk_plot_df['metric'].isin(['failed_payment_revenue', 'successful_payment_revenue'])]

plt.figure(figsize=(8, 5))
plt.bar(risk_plot_df['metric'], risk_plot_df['amount_usd'], color=['#E15759', '#59A14F'])
plt.title('Revenue at Risk vs Successful Revenue')
plt.ylabel('USD')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## Business Insights (Plain English)

In [ ]:
# Print 5 business insights in plain English.
search_to_booking_conv = float(funnel_df.loc[funnel_df['stage'] == 'bookings', 'conversion_from_search_pct'].iloc[0])
payment_success_rate = float((payment['payment_status'] == 'success').mean() * 100)
ticket_issue_rate_after_success = float((ticket_after_success['ticket_status'] == 'issued').mean() * 100)
refund_request_rate_success = float((refund['refund_requested'].mean()) * 100)
top_error = payment_failure_by_error_code.iloc[0]['payment_error_code'] if not payment_failure_by_error_code.empty else 'N/A'

insights = [
    f'1) The largest funnel drop happens before booking: only {search_to_booking_conv:.2f}% of searches convert to bookings.',
    f'2) Payment reliability is strong but still a meaningful leakage point: payment success is {payment_success_rate:.2f}% and failed-payment revenue is ${failed_payment_revenue:,.2f}.',
    f'3) Fulfillment quality after successful payment is high: {ticket_issue_rate_after_success:.2f}% of successful payments result in issued tickets.',
    f'4) Refund activity is concentrated in a minority of successful payments: refund requested rate is {refund_request_rate_success:.2f}%, which helps bound downstream revenue leakage.',
    f'5) The most frequent payment failure code is {top_error}, making it a high-priority candidate for root-cause analysis and optimization.'
]

for line in insights:
    print(line)